<a href="https://colab.research.google.com/github/unatisaini/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/unatisaini/flyrank-internship-mi/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup: load the lane data

Same data and preparation as my capstone notebook (`w08_capstone.ipynb`). It needs a Hugging Face read token: add it once as a Colab secret named `HF_TOKEN` (key icon in the left sidebar, then switch on notebook access), or paste it when asked.

In [1]:
!pip -q install pandas pyarrow scikit-learn matplotlib huggingface_hub
import numpy as np, pandas as pd
from huggingface_hub import login, hf_hub_download

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    from getpass import getpass
    login(token=getpass("Paste your Hugging Face read token: "))

path = hf_hub_download(repo_id="FlyRank/internship-lanes",
                       filename="default_lanes/engagement_fix.parquet",
                       repo_type="dataset")
df = pd.read_parquet(path)
print("rows, columns:", df.shape)

Paste your Hugging Face read token: ··········


default_lanes/engagement_fix.parquet: reconstructing file:   0%|          |  0.00B /  760kB            

default_lanes/engagement_fix.parquet: downloading bytes:           |  0.00B            

rows, columns: (33202, 31)


In [2]:
# Same preparation as the capstone: visible pages, position bands, feature sets, label rule
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

d = df.dropna(subset=["ctr_30d", "avg_position_30d"]).copy()
a = d[d.impressions_30d >= 100].copy().reset_index(drop=True)

bins = [0, 3, 5, 7, 10, 15, 20, 30, 200]
a["pos_band"] = pd.cut(a.avg_position_30d, bins, include_lowest=True).astype(str)
a["log_impr"] = np.log1p(a.impressions_30d)

POS = ["avg_position_30d", "log_impr"]
ENG = ["engagement_rate_30d", "scroll_rate_30d", "ai_traffic_pct_30d"]
VOL = ["pageviews_30d", "sessions_30d", "users_30d",
       "engaged_sessions_30d", "ai_sessions_30d", "scroll_events_30d"]
CAT = ["content_type", "main_intent", "age_tier", "freshness_tier",
       "word_count_tier", "char_count_tier"]
for c in CAT:
    a[c] = a[c].astype("category")
FEATURES = POS + ENG + CAT

def make_labels(train, test):
    """Under-performer = CTR below half the TRAIN-fold median CTR of the page's position band."""
    med = train.groupby("pos_band").ctr_30d.median()
    out = []
    for part in (train, test):
        exp = part.pos_band.map(med).astype(float)
        out.append(((part.ctr_30d / exp) < 0.5).astype(float).where(exp > 0))
    return out

def new_model():
    return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.08, max_iter=200,
                                          categorical_features="from_dtype", random_state=42)

def folds():
    return GroupKFold(n_splits=5).split(a, groups=a.client_hash_id)

print("all rows:", len(df), "| with CTR + position:", len(d), "| visible (>= 100 impressions):", len(a))
print("clients with visible pages:", a.client_hash_id.nunique())

all rows: 33202 | with CTR + position: 32844 | visible (>= 100 impressions): 28078
clients with visible pages: 38


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page (`content_hash_id`) of one client (`client_hash_id`), with 30-day totals.

- **Data:** the CTR/engagement lane cut `default_lanes/engagement_fix.parquet` from the gated `FlyRank/internship-lanes` dataset, published with warehouse build `flyrank_pseudonymized_warehouse_release_v20260703` (exported 2026-07-03).
- **Time window:** one 30-day window. The cut has no date column, so the exact dates can't be stated. It lies inside the release, whose daily data end on 2026-06-30.
- My first draft of this contract planned a page × query monthly table. For the capstone I switched to this page-level lane cut, so the contract below describes the data I actually used.

In [3]:
print("rows:", len(df), "| columns:", df.shape[1])
print("clients:", df.client_hash_id.nunique())
print("max rows per (client, page) id:", df.groupby(["client_hash_id", "content_hash_id"]).size().max())
print("fully duplicated rows:", df.duplicated().sum())
date_cols = [c for c in df.columns if "date" in c.lower() or "month" in c.lower()]
print("date-like columns:", date_cols or "none")

rows: 33202 | columns: 31
clients: 44
max rows per (client, page) id: 1
fully duplicated rows: 0
date-like columns: none


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature:** `avg_position_30d`, `impressions_30d` (used as log), `engagement_rate_30d`, `scroll_rate_30d`, `ai_traffic_pct_30d`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `char_count_tier`
- **Label source (builds the label, never a feature):** `ctr_30d`, `clicks_30d`
- **Context:** `client_hash_id` (groups the validation split), `content_hash_id` (row id), `client_has_gsc` / `client_has_ga4` (true for every row), `impression_tier` / `position_tier` (tiers of columns I already use)
- **Excluded, product rules:** `health_score`, `needs_indexing`, `is_quick_win`, `needs_ctr_fix`, `needs_engagement_fix`, `ai_opportunity`. These are FlyRank's own rule outputs, not observations. `needs_ctr_fix` is used only as a baseline to beat.
- **Excluded, tracks clicks:** `pageviews_30d`, `sessions_30d`, `users_30d`, `engaged_sessions_30d`, `ai_sessions_30d`, `scroll_events_30d`. Visits start as clicks, so these follow the label. They are used only in a sensitivity check.

In [4]:
buckets = {
    "feature": ["avg_position_30d", "impressions_30d", "engagement_rate_30d", "scroll_rate_30d",
                "ai_traffic_pct_30d", "content_type", "main_intent", "age_tier", "freshness_tier",
                "word_count_tier", "char_count_tier"],
    "label source": ["ctr_30d", "clicks_30d"],
    "context": ["client_hash_id", "content_hash_id", "client_has_gsc", "client_has_ga4",
                "impression_tier", "position_tier"],
    "excluded: product rule": ["health_score", "needs_indexing", "is_quick_win", "needs_ctr_fix",
                               "needs_engagement_fix", "ai_opportunity"],
    "excluded: tracks clicks": ["pageviews_30d", "sessions_30d", "users_30d",
                                "engaged_sessions_30d", "ai_sessions_30d", "scroll_events_30d"],
}
rows = [(col, b, str(df[col].dtype), int(df[col].isna().sum()))
        for b, cols in buckets.items() for col in cols]
contract = pd.DataFrame(rows, columns=["column", "bucket", "dtype", "missing"])
print("columns not sorted into a bucket:", sorted(set(df.columns) - set(contract.column)) or "none")
contract

columns not sorted into a bucket: none


,column,bucket,dtype,missing
0,avg_position_30d,feature,float64,358
1,impressions_30d,feature,int64,0
2,engagement_rate_30d,feature,float64,0
3,scroll_rate_30d,feature,float64,0
4,ai_traffic_pct_30d,feature,float64,0
5,content_type,feature,object,0
6,main_intent,feature,object,0
7,age_tier,feature,object,0
8,freshness_tier,feature,object,0
9,word_count_tier,feature,object,5680


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every claim above is checked in code below: row counts at each filter step, the CTR unit, client balance and missing values. My capstone run gave 33,202 rows, then 32,844 with CTR and position, then 28,078 with at least 100 impressions, from 38 clients. The largest client holds 32% of visible pages, CTR matches clicks ÷ impressions × 100 to within 0.005 points, and the word/char count tiers are blank for 5,680 rows.

In [5]:
d_chk = df.dropna(subset=["ctr_30d", "avg_position_30d"])
vis = d_chk[d_chk.impressions_30d >= 100]
print("all rows:", len(df))
print("with CTR and position:", len(d_chk), f"(dropped {len(df) - len(d_chk)})")
print("with >= 100 impressions:", len(vis), f"(dropped {len(d_chk) - len(vis)})")
print("clients among visible pages:", vis.client_hash_id.nunique())
print("largest client share of visible pages:", round(vis.client_hash_id.value_counts(normalize=True).iloc[0], 3))

ctr_calc = d_chk.clicks_30d / d_chk.impressions_30d.replace(0, np.nan) * 100
print("CTR unit check, max |ctr_30d - clicks/impressions*100|:", round((d_chk.ctr_30d - ctr_calc).abs().max(), 4))

print("\nmissing values (columns with any):")
print(df.isna().sum()[lambda s: s > 0].to_string())

all rows: 33202
with CTR and position: 32844 (dropped 358)
with >= 100 impressions: 28078 (dropped 4766)
clients among visible pages: 38
largest client share of visible pages: 0.322
CTR unit check, max |ctr_30d - clicks/impressions*100|: 0.005

missing values (columns with any):
word_count_tier     5680
char_count_tier     5680
ctr_30d              358
avg_position_30d     358


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **One 30-day window, no dates.** I can't measure trends or before/after effects, and the label comes from the same window as the features.
- **Average position is a page-level average** over many queries, so it hides the query mix behind each page.
- **Engagement and scroll rates are never blank**, even for pages with no sessions. A 0 there can mean "no visits" rather than "visitors didn't engage".
- **One content type dominates** (99.6% of visible pages are keyword articles), so content type can't carry much signal.
- **Clients are very unbalanced.** The median client has about 58 pages and the largest has 9,400.
- **Product flags are in the file, but they are FlyRank decisions, not observations.**

In [6]:
print("content type share (visible pages):")
print(vis.content_type.value_counts(normalize=True).round(3).to_string())
print("\npages per client (all rows):")
print(df.groupby("client_hash_id").size().describe().round(1).to_string())
zero = df[df.sessions_30d == 0]
print("\npages with zero sessions:", len(zero))
print("engagement_rate_30d on those pages:", zero.engagement_rate_30d.describe().round(2).to_dict())

content type share (visible pages):
content_type
keyword article    0.996
feedly article     0.004

pages per client (all rows):
count      44.0
mean      754.6
std      2014.0
min         1.0
25%         8.5
50%        57.5
75%       437.2
max      9400.0

pages with zero sessions: 0
engagement_rate_30d on those pages: {'count': 0.0, 'mean': nan, 'std': nan, 'min': nan, '25%': nan, '50%': nan, '75%': nan, 'max': nan}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.